### Convert embeddings raster to "expert" patch embeddings

"Expert" embeddings assemble mean, std, min, max embedding values across a patch and concatenate into a single 
 long embedding vector. 

Notebook outputs a GeoDataFrame where each row contains the expert embedding and the centroid point geometry for the patch. 

In [1]:
from glob import glob
from pathlib import Path
import sys
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio

try:
    _root = Path(__file__).resolve().parent.parent
except NameError:
    _root = Path.cwd()
    if not (_root / "gee" / "tile_utils.py").exists() and (_root.parent / "gee" / "tile_utils.py").exists():
        _root = _root.parent
sys.path.insert(0, str(_root / "gee"))
from tile_utils import cut_chips

Adjust the path following to point to a collection of GeoTiffs of embedding vectors, which might have been downloaded from Earth Engine with `gee_data_pull.py`

In [ ]:
raster_embeddings_path = Path('/path/to/my/embeddings/')
paths = sorted(glob(str(raster_embeddings_path / '*.tif')))
if not paths:
    raise FileNotFoundError("No .tif files found. Set raster_embeddings_path to your embeddings directory.")

In [ ]:
# It might be interesting to experiment with different sets of statistics here:

def expert_stats(patches, axis=(2,3)): 
    """Compute statistics for expert embeddings. 
    
    patches: ndarray of shape (N, C, H, W)
    axis: tuple: Axes over which to take summary statistics.
    """
    means = patches.mean(axis=axis)  
    stds = patches.std(axis=axis)
    maxes = patches.max(axis=axis)
    mins = patches.min(axis=axis)
    return means, stds, maxes, mins

n_stats = len(expert_stats(np.zeros((1,1,1,1))))
n_stats

In [ ]:
patch_size = 8 
stride = 8  

with rasterio.open(paths[0]) as f:
    embedding_dim = f.count  
    
feature_columns = [f"e{i}" for i in range(embedding_dim * n_stats)]
gdfs = []

for path in paths: 
    
    with rasterio.open(path) as f:
        bounds = f.bounds
        crs = f.crs
        img = f.read()
        
    assert img.shape[0] == embedding_dim
        
    img = np.moveaxis(img, 0, -1)
    patches, geoms = cut_chips(img, bounds, chip_size=patch_size, stride=stride, crs=crs)
    patches = np.moveaxis(patches, -1, 1)
    
    expert = np.array(expert_stats(patches))
    expert = np.moveaxis(expert, 0, -1)
    N, C, S = expert.shape
    expert = expert.reshape(N, C * S)
        
    df = pd.DataFrame(expert, columns=feature_columns)
    gdf = gpd.GeoDataFrame(df, geometry=geoms['geometry'])
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, message="Geometry is in a geographic CRS")
        gdf['geometry'] = gdf.geometry.centroid
    gdfs.append(gdf)
    
combined_gdf = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    geometry="geometry",
    crs=crs
)

print(f"{len(combined_gdf)} expert embeddings created from {len(paths)} files")
combined_gdf.head()

In [ ]:
combined_gdf.to_parquet(raster_embeddings_path / f"patch_size{patch_size}_stride{stride}_embeddings.parquet", index=False)
